In [3]:
import pandas as pd

file_path = '/Users/samkhatri/Desktop/Data Engineering/ShopStream-Realtime-BigData-Ecommerce-pipeline/Data/online_retail_II.xlsx'

df1 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
df2 = pd.read_excel(file_path, sheet_name='Year 2010-2011')

In [6]:
# Adding the year column to each
df1['year'] = 2009
df2['year'] = 2010

combined_df = pd.concat([df1, df2], ignore_index=True)



In [9]:
len(combined_df)

1067371

In [22]:
combined_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,year,month
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009,12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009,12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009,12


In [20]:
combined_df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
year                    int64
dtype: object

In [21]:
combined_df['month'] = combined_df['InvoiceDate'].dt.month

In [24]:
combined_df['Customer ID'].nunique()

5942

In [25]:
import pandas as pd
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

In [26]:
API_ENDPOINT = "XXXX"

In [36]:
import math
import time

# --- CONFIGURATION ---
total_records = len(combined_df)
total_duration = 300  # 5 minutes
chunk_size = 2000
max_workers = 50
retries = 3

batches = math.ceil(total_records / chunk_size)
sleep_time = total_duration / batches

print(f"📦 Sending {total_records} records in ~{total_duration} seconds")
print(f"🧩 {chunk_size} per batch, {batches} batches, every {sleep_time:.2f} sec")

# Open file to log failures
failure_log = open("failed_records.jsonl", "w")

def send_transaction(row):
    data = {
        "invoice": str(row["Invoice"]),
        "stock_code": row["StockCode"],
        "description": row["Description"],
        "quantity": int(row["Quantity"]),
        "timestamp": row["InvoiceDate"].strftime("%Y-%m-%dT%H:%M:%S"),
        "price": float(row["Price"]),
        "customer_id": str(row["Customer ID"]),
        "country": row["Country"],
        "year": int(row["year"]),
        "month": int(row["month"])
    }

    for attempt in range(retries):
        try:
            response = requests.post(API_ENDPOINT, json=data, timeout=3)
            if response.status_code == 200:
                return 200
            else:
                time.sleep(0.5 * (attempt + 1))  # exponential backoff
        except Exception:
            time.sleep(0.5 * (attempt + 1))
    
    # Log failed record for retry later
    failure_log.write(json.dumps(data) + "\n")
    return "❌ Failed"

def stream_data_in_parallel(df):
    success_count = 0
    fail_count = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(send_transaction, row) for _, row in df.iterrows()]
        for future in as_completed(futures):
            result = future.result()
            if result == 200:
                success_count += 1
            else:
                fail_count += 1
            print(result)

    print(f"✅ Batch Complete: {success_count} succeeded, {fail_count} failed\n")

# --- STREAMING LOOP ---
for i in range(0, total_records, chunk_size):
    chunk = combined_df.iloc[i:i + chunk_size]
    print(f"🚀 Sending batch {i // chunk_size + 1}: {len(chunk)} records")
    stream_data_in_parallel(chunk)
    time.sleep(sleep_time)

failure_log.close()